# 72장 · Docker Compose 네트워크와 볼륨

같은 DB에 호스트에서는 55432로 접속하는데 api 컨테이너에서는 왜 5432를 쓸까요?

호스트에서 PostgreSQL에 접속하는 주소와 API 컨테이너에서 사용하는 주소를 비교합니다. Compose의 서비스 이름, 내부 포트, 호스트 포트, 볼륨이 각각 어디에 쓰이는지 확인합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 PyYAML 6.0.3입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 서비스 이름과 내부 포트

호스트 컴퓨터에서 보는 127.0.0.1과 API 컨테이너 안에서 보는 127.0.0.1은 각자의 네트워크 공간입니다. Compose의 공통 네트워크에서는 API가 db라는 서비스 이름과 DB의 컨테이너 포트 5432로 접속합니다.


### API에서 보이는 주소와 호스트에서 보이는 주소

<p>Compose 네트워크의 API는 PGHOST에서 db를, PGPORT에서 5432를 읽습니다. 호스트 쪽 공개 주소는 127.0.0.1:55432입니다. 어디에서 접속하는지가 달라지면 같은 DB를 가리키는 주소도 달라집니다.</p>

<h3>Compose YAML에서 실제로 공개한 포트</h3>

<p>db의 ports에만 127.0.0.1:55432:5432가 있습니다. 호스트의 DB 클라이언트는 127.0.0.1:55432, 내부 api는 db:5432로 연결합니다. api에는 ports 항목이 없어 이 선언만으로 호스트의 8000번 포트에서 API에 접속할 수 없습니다.</p>


### 셀 01 · Compose 서비스 이름과 포트

db 서비스의 ports와 api 서비스의 environment를 비교합니다. yaml.safe_load가 두 설정을 딕셔너리로 읽으므로 Docker가 없어도 선언한 주소는 확인할 수 있습니다.


In [1]:
import yaml
compose_text = """name: course-b1
services:
  db:
    image: pgvector/pgvector:pg16
    ports: ["127.0.0.1:55432:5432"]
    volumes: ["course-data:/var/lib/postgresql/data"]
  api:
    image: course-api:classroom
    environment:
      PGHOST: db
      PGPORT: "5432"
volumes:
  course-data: {}
"""
config = yaml.safe_load(compose_text)
connection = config["services"]["api"]["environment"]
print(connection["PGHOST"], connection["PGPORT"])

db 5432


db 5432는 api 컨테이너의 환경 설정에서 읽은 두 값입니다. 호스트에서 노트북을 실행할 때는 db라는 서비스 이름 대신 호스트에 공개된 IP와 포트를 사용합니다.


#### 같은 DB인데 접속 주소가 다른 이유

<mark>다른 컨테이너에서는 서비스 이름과 컨테이너 쪽 포트를 사용합니다.</mark> 호스트에서 쓰는 127.0.0.1:55432를 앱 컨테이너 안에 그대로 넣으면 자기 자신을 가리킵니다. 연결을 여는 프로그램이 어디에서 실행되는지 먼저 표시하세요.

선수 개념 · 71장 Dockerfile과 컨테이너 수명 주기

### Compose 구성 검사에서 확인한 값

<p>docker compose config는 YAML을 해석해 정규화된 설정을 출력합니다. 여기서 DB의 target 경로를 읽은 뒤, 파일을 저장하고 다시 여는 동작은 다음 SQLite 셀에서 따로 실험합니다.</p>


### 과제 02 · 호스트 포트 매핑

호스트에서 실행한 DB 도구와 API 컨테이너에서 사용할 접속 주소를 각각 구성하세요. 두 주소의 호스트 이름과 포트가 모두 달라야 합니다.


In [2]:
port_map = config['services']['db']['ports'][0]
# 예제의 IPv4 공개 포트 표기와 connection 설정에서 두 주소를 추출하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>port_map = config[&quot;services&quot;][&quot;db&quot;][&quot;ports&quot;][0]
host, published_port, container_port = port_map.split(&quot;:&quot;)
print(f&quot;{host}:{published_port}&quot;)
print(f&quot;{connection[&#x27;PGHOST&#x27;]}:{connection[&#x27;PGPORT&#x27;]}&quot;)</code></pre>
<p>실행 결과</p><pre><code>127.0.0.1:55432
db:5432</code></pre>
<p>첫 주소는 호스트에서 열린 포트이고 둘째는 Compose 네트워크 안에서 DB 서비스가 받는 주소입니다. 공개 포트만 55433으로 바꾸면 첫 줄만 달라져야 합니다.</p>
</details>


## 볼륨과 컨테이너 교체

선언된 볼륨은 DB의 데이터 디렉터리에 연결됩니다. 같은 컴퓨터·프로젝트의 같은 볼륨을 다시 연결해야 기존 자료를 읽을 수 있습니다. 다른 호스트에 같은 이름만 만들어도 이전 데이터가 생기는 것은 아닙니다.


### 볼륨 이름과 컨테이너 경로

<p>course-data는 Compose에 선언한 볼륨 키이고 /var/lib/postgresql/data는 DB 컨테이너 안의 연결 위치입니다. 둘을 구분하면 컨테이너를 교체할 때 어느 저장소를 다시 연결할지 읽을 수 있습니다.</p>


### 셀 03 · Compose 구성 검증

Docker Compose가 설치된 커널에서는 실제 config 명령이 선언을 해석합니다. 이미지를 받거나 컨테이너를 띄우지 않습니다.


In [3]:
import json, shutil, subprocess
if shutil.which("docker"):
    checked = subprocess.run(["docker", "compose", "-f", "-", "config", "--format", "json"],
                             input=compose_text, text=True, capture_output=True, timeout=20)
    if checked.returncode == 0:
        normalized = json.loads(checked.stdout)
        print(normalized["services"]["db"]["volumes"][0]["target"])
    else:
        print("Compose 구성을 확인하지 못했습니다.")
else:
    print("Docker Compose CLI가 없어 선언 검사를 건너뜁니다.")

/var/lib/postgresql/data


CLI가 구성 해석에 성공하면 /var/lib/postgresql/data가 보입니다. 이 경로가 컨테이너 안에서 볼륨을 연결할 위치입니다. 실제로 재시작한 뒤 같은 문서가 남았는지는 데이터베이스에서 다시 조회해야 합니다.


## 연결을 다시 열어 저장 데이터 검증

볼륨 위에 DB가 유지된다는 말은 새 연결에서 원래 행을 다시 읽을 수 있다는 뜻입니다. 아래는 임시 파일 하나에 실제 SQLite DB를 만들고 연결을 닫았다가 다시 여는 연습입니다. Docker 컨테이너와 PostgreSQL의 재시작은 이 코드에서 일어나지 않습니다.


### 셀 04 · DB 재연결과 영속성 확인

행 수만 비교하면 내용이 달라져도 알아채기 어렵습니다. 같은 소유자·문서 ID로 원문을 읽고 대조하세요.

셀 03은 Compose 설정에서 연결 대상을 읽습니다. 셀 04가 다시 여는 대상은 SQLite 파일입니다. 실제 볼륨 보존을 확인하려면 같은 볼륨을 연결한 DB 컨테이너를 다시 띄우고 이전 행을 조회해야 합니다.


In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory
import sqlite3

with TemporaryDirectory() as work:
    path = Path(work) / "course.db"
    db = sqlite3.connect(path)
    db.execute("CREATE TABLE docs(owner TEXT, id TEXT, body TEXT, PRIMARY KEY(owner,id))")
    db.execute("INSERT INTO docs VALUES (?,?,?)", ("alice", "doc-1", "과제함에 제출합니다."))
    db.commit()
    lookup = "SELECT body FROM docs WHERE owner=? AND id=?"
    key = ("alice", "doc-1")
    before = db.execute(lookup, key).fetchone()
    db.close()
    reopened = sqlite3.connect(path)
    after = reopened.execute(lookup, key).fetchone()
    reopened.close()
print(after[0], before == after)

과제함에 제출합니다. True


과제함 문장과 True가 함께 나옵니다. 실제 Compose 환경에서는 재시작 전후 같은 요청을 보내 이 세 값, 즉 소유자·문서 ID·원문을 비교합니다. 이전 DB 볼륨이 연결되지 않았다면 이 조회에서 원래 행을 찾을 수 없습니다. 볼륨을 확인할 때는 수업용 Compose 환경에서 한 행의 ID를 기록한 뒤 같은 볼륨을 연결해 DB 서비스를 재시작하세요. 그 ID로 다시 조회한 값을 SQLite 실습 결과와 따로 적으세요.


#### 연결을 닫은 뒤에도 같은 본문을 읽는 경로

```mermaid
flowchart TD
nwrite["INSERT"]
  nsave["commit"]
  nclose["close"]
  nreopen["connect"]
  nselect["SELECT"]
  nwrite --> nsave
  nsave --> nclose
  nclose --> nreopen
  nreopen --> nselect
```

이 셀은 임시 SQLite 파일로 재연결을 실험합니다. Compose 볼륨의 실제 재생성은 별도 검증 대상입니다.

### 조회 조건을 고정한 재연결 검사

<p>같은 파일에서 재연결 전과 후에 같은 owner와 id로 본문을 읽습니다. before == after가 True라는 출력은 이 경로와 이 키에서 값이 유지되었다는 뜻입니다.</p>


## DB 준비를 기다리는 Compose 의존성 작성하기

앞에서는 서비스 이름과 볼륨 경로를 읽었습니다. 이번에는 DB 컨테이너가 시작된 직후 API가 먼저 접속하는 상황을 다룹니다. 실행 순서와 DB의 준비 상태를 Compose 파일에 어떻게 적는지 확인합니다.

### 셀 06 · DB의 준비 상태를 확인할 명령 추가하기

셀 01에서 compose_text를 읽어 config를 만들어 두세요. 앞에서 읽은 Compose 구성을 복사한 뒤 db 서비스에 준비 상태 검사 명령을 붙입니다. 결과는 메모리 안에 만든 Compose 설정입니다.

In [5]:
from copy import deepcopy

ready_config = deepcopy(config)
ready_config["services"]["db"]["healthcheck"] = {
    "test": ["CMD-SHELL", "pg_isready -h 127.0.0.1 -U postgres -d postgres"],
    "interval": "5s", "timeout": "3s", "retries": 10,
}
print(ready_config["services"]["db"]["healthcheck"]["test"])

['CMD-SHELL', 'pg_isready -h 127.0.0.1 -U postgres -d postgres']


CMD-SHELL은 검사 명령을 db 컨테이너 안에서 실행할 형식입니다. pg_isready는 서버가 연결을 받을 수 있는지를 확인합니다. 테이블이나 실습 문서가 준비됐는지는 여기에 포함되지 않습니다.

### 과제 07 · API가 기다릴 조건을 Compose YAML로 만들기

api 서비스의 depends_on에서 db에 service_healthy 조건을 주고, 실제로 Compose에 전달할 YAML을 만드세요. 직렬화한 문자열을 다시 열어 API가 기다리는 서비스·조건을 출력하세요.

In [6]:
from copy import deepcopy
import yaml
team_stack = deepcopy(ready_config)
# api 서비스 아래에 depends_on을 추가하고 YAML로 저장할 문자열을 만드세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>from copy import deepcopy
import yaml

team_stack = deepcopy(ready_config)
team_stack[&quot;services&quot;][&quot;api&quot;][&quot;depends_on&quot;] = {&quot;db&quot;: {&quot;condition&quot;: &quot;service_healthy&quot;}}
compose_input = yaml.safe_dump(team_stack, sort_keys=False)
loaded_stack = yaml.safe_load(compose_input)
for service, rule in loaded_stack[&quot;services&quot;][&quot;api&quot;][&quot;depends_on&quot;].items():
    print(service, rule[&quot;condition&quot;])</code></pre>
<p>실행 결과</p><pre><code>db service_healthy</code></pre>
<p>db service_healthy가 나옵니다. db가 컨테이너 목록에 나타난 시점보다 늦게 API를 시작할 수 있게 했습니다. 수업용 DB의 계정 설정과 API 이미지가 준비된 환경에서는 앞 절의 docker compose config 검사에 compose_input을 넣고, 실제 기동 후에는 저장 자료까지 읽어 보세요.</p>
</details>

### 셀 08 · 선택 실습 · 컨테이너를 교체한 뒤 같은 DB 행 읽기

로컬 Docker 엔진, Compose의 --wait 지원, 앞 예제의 pgvector 이미지가 준비됐다면 run_stack을 켜세요. 외부 포트를 열지 않는 임시 프로젝트에서 한 행을 저장하고 DB 컨테이너만 교체합니다. 마지막에 이 프로젝트의 수업용 볼륨을 지웁니다.

In [7]:
import secrets
import subprocess
import yaml
from uuid import uuid4

run_stack = False
if run_stack:
    project = "lesson72-" + uuid4().hex[:10]
    spec = {"services": {"db": {
        "image": ready_config["services"]["db"]["image"],
        "environment": {"POSTGRES_PASSWORD": secrets.token_urlsafe(24)},
        "healthcheck": ready_config["services"]["db"]["healthcheck"],
        "volumes": ["classdata:/var/lib/postgresql/data"],
    }}, "volumes": {"classdata": {}}}
    def compose_cmd(*args):
        cmd = ["docker", "compose", "-p", project, "-f", "-", *args]
        return subprocess.run(cmd, input=yaml.safe_dump(spec), check=True,
                              text=True, capture_output=True, timeout=120).stdout.strip()
    def query_db(sql):
        return compose_cmd("exec", "-T", "db", "psql", "-U", "postgres",
                           "-d", "postgres", "-At", "-v", "ON_ERROR_STOP=1", "-c", sql)
    try:
        compose_cmd("up", "-d", "--pull", "never", "--wait", "db")
        query_db("CREATE TABLE notes(id TEXT PRIMARY KEY)")
        query_db("INSERT INTO notes VALUES ('course-guide')")
        first_id = compose_cmd("ps", "-q", "db")
        compose_cmd("up", "-d", "--pull", "never", "--force-recreate", "--wait", "db")
        print(first_id != compose_cmd("ps", "-q", "db"), query_db("SELECT id FROM notes"))
    finally:
        compose_cmd("down", "--volumes")
else:
    print("로컬 이미지와 Compose가 준비되면 run_stack을 켜세요.")

로컬 이미지와 Compose가 준비되면 run_stack을 켜세요.


True course-guide가 관찰되면 컨테이너 ID가 바뀌어도 같은 볼륨에서 이전 행을 읽은 것입니다. 기본값에서는 엔진이나 DB에 연결하지 않습니다. 이전 절의 설정 검사는 이 저장 실험을 대신하지 않습니다.

<mark>DB의 준비 상태와 저장한 행의 보존 여부는 서로 다른 요청으로 검사합니다.</mark> pg_isready가 성공한 빈 DB에서도 과제 안내 검색은 실패할 수 있습니다. 재시작 전 문서 ID와 본문을 적어 두고 재시작 후 같은 요청으로 조회해야 볼륨도 확인할 수 있습니다.

API 이미지는 71장에서 만든 것을, DB에는 수업에서 준비한 계정 설정을 사용합니다. 볼륨과 계정을 연결한 뒤 73장의 벡터 검색을 시험하세요.

## 돌아보기

SQLite 파일을 다시 연 뒤 같은 문장이 나와도 Compose 볼륨을 검증한 것은 아닙니다. 실제 DB 컨테이너를 교체할 때는 먼저 같은 볼륨이 연결됐는지 보고, 교체 전 문서 ID로 본문을 다시 읽습니다.
